# 🤖 01 - Train Insurance Models (MLflow)

## Objective
Train and compare multiple regression models to predict `charges`, using a single preprocessing pipeline and tracking all experiments in **MLflow**.

**Outputs:**
- MLflow runs (params, metrics, artifacts)
- Best model saved locally (`models/best_model.pkl`)
- Evaluation plots saved in `reports/figures/`

## 1) Imports

In [30]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import joblib

import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

## 2) Paths y configuración

In [32]:
# Project paths (si corres desde notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # notebooks/ -> root
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "insurance", "insurance.csv")

MLRUNS_DIR = os.path.join(PROJECT_ROOT, "mlruns")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "reports", "figures")

os.makedirs(MLRUNS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

mlflow.set_tracking_uri("file:" + MLRUNS_DIR)
mlflow.set_experiment("insurance_charges_prediction")

DATA_PATH

'd:\\Maestria DS\\Ciclo III\\C03_MLOps\\Project_example\\mlops-final-project-Jonathan_Sanchez\\data\\insurance\\insurance.csv'

## 3) Cargar data y validaciones rápidas

In [33]:
assert os.path.exists(DATA_PATH), f"No existe el archivo en: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df.head(), df.shape

(   age     sex     bmi  children smoker     region      charges
 0   19  female  27.900         0    yes  southwest  16884.92400
 1   18    male  33.770         1     no  southeast   1725.55230
 2   28    male  33.000         3     no  southeast   4449.46200
 3   33    male  22.705         0     no  northwest  21984.47061
 4   32    male  28.880         0     no  northwest   3866.85520,
 (1338, 7))

## 4) Definir target + split

In [34]:
target = "charges"
assert target in df.columns, f"No existe la columna target '{target}'"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((1070, 6), (268, 6))

## 5) Preprocesamiento

In [35]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

cat_cols, num_cols

(['sex', 'smoker', 'region'], ['age', 'bmi', 'children'])

## 6) Definir modelos a comparar

In [36]:
models = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0, random_state=42),
    "lasso": Lasso(alpha=0.001, random_state=42),
    "elasticnet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42),

    "random_forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1, max_depth=None
    ),

    "gboost": GradientBoostingRegressor(random_state=42),

    # SVR suele ser más lento; útil para comparar
    "svr_rbf": SVR(C=10.0, epsilon=0.1, kernel="rbf"),
}

list(models.keys())

['linear_regression',
 'ridge',
 'lasso',
 'elasticnet',
 'random_forest',
 'gboost',
 'svr_rbf']

## 7) Config CV + funciones de métricas

In [37]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

def eval_on_test(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

## 8) Setup MLflow + Experimento

In [39]:
import os
import mlflow
import mlflow.sklearn

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # si estás en notebooks/
MLRUNS_DIR = os.path.join(PROJECT_ROOT, "mlruns")

mlflow.set_tracking_uri("file:" + MLRUNS_DIR)
mlflow.set_experiment("insurance_charges_prediction")

<Experiment: artifact_location=('file:d:\\Maestria DS\\Ciclo '
 'III\\C03_MLOps\\Project_example\\mlops-final-project-Jonathan_Sanchez\\mlruns/627980517703590058'), creation_time=1772179370561, experiment_id='627980517703590058', last_update_time=1772179370561, lifecycle_stage='active', name='insurance_charges_prediction', tags={}, workspace='default'>

## 9) Métricas helper (RMSE sin squared=False)

In [40]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

## 10) Modelos a probar (rápidos y típicos)

In [41]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

models = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0, random_state=42),
    "lasso": Lasso(alpha=0.001, random_state=42, max_iter=10000),
    "random_forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "gbr": GradientBoostingRegressor(random_state=42),
    "svr": SVR(C=10.0, epsilon=0.1, kernel="rbf"),
}

## 11) CV (cross-validation) sobre el set de entrenamiento

In [42]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",  # si tu sklearn no lo soporta, te doy alternativa
    "r2": "r2"
}

## 12) Loop de entrenamiento + logging en MLflow (CV + Test + modelo)

In [43]:
import pandas as pd

results = []
best = {"model": None, "name": None, "cv_rmse": np.inf, "run_id": None}

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    with mlflow.start_run(run_name=model_name) as run:
        run_id = run.info.run_id

        # --- log params básicos
        mlflow.log_param("model_name", model_name)

        # log params específicos si existen
        if hasattr(model, "get_params"):
            params = model.get_params()
            # no loguees todo si es enorme; filtra los más relevantes
            for k in ["alpha", "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf",
                      "random_state", "C", "epsilon", "kernel"]:
                if k in params:
                    mlflow.log_param(k, params[k])

        # --- CV
        cv_out = cross_validate(
            pipe, X_train, y_train,
            cv=cv, scoring=scoring,
            return_train_score=False,
            n_jobs=-1
        )

        cv_mae = -cv_out["test_mae"].mean()
        # Caso 1: si existe rmse scoring:
        if "test_rmse" in cv_out:
            cv_rmse = -cv_out["test_rmse"].mean()
        else:
            # Caso 2: si usamos mse scoring
            cv_mse = -cv_out["test_mse"].mean()
            cv_rmse = np.sqrt(cv_mse)

        cv_r2 = cv_out["test_r2"].mean()

        mlflow.log_metric("cv_mae", float(cv_mae))
        mlflow.log_metric("cv_rmse", float(cv_rmse))
        mlflow.log_metric("cv_r2", float(cv_r2))

        # --- Fit final + Test
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        test_mae, test_rmse, test_r2 = eval_metrics(y_test, preds)

        mlflow.log_metric("test_mae", float(test_mae))
        mlflow.log_metric("test_rmse", float(test_rmse))
        mlflow.log_metric("test_r2", float(test_r2))

        # --- Log model (ojo con warning de artifact_path vs name)
        mlflow.sklearn.log_model(pipe, name="model")

        results.append({
            "model": model_name,
            "cv_mae": cv_mae,
            "cv_rmse": cv_rmse,
            "cv_r2": cv_r2,
            "test_mae": test_mae,
            "test_rmse": test_rmse,
            "test_r2": test_r2,
            "run_id": run_id
        })

        # --- tracking del mejor (por menor cv_rmse)
        if cv_rmse < best["cv_rmse"]:
            best.update({"model": pipe, "name": model_name, "cv_rmse": cv_rmse, "run_id": run_id})

df_results = pd.DataFrame(results).sort_values("cv_rmse")
df_results

2026/02/27 03:45:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/27 03:45:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/27 03:45:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

,model,cv_mae,cv_rmse,cv_r2,test_mae,test_rmse,test_r2,run_id
4,gbr,2611.103313,4646.314895,0.849230,2404.901760,4328.147789,0.879336,f3b90d18373445109f15eaee542dd11a
3,random_forest,2743.685581,4882.498571,0.834013,2525.646339,4603.868639,0.863473,fa60911a146843fda55c172555fe6adf
1,ridge,4240.596409,6123.344389,0.738855,4186.913072,5798.298795,0.783443,57dff8c929bd4439baba3d6b4cfe36e4
2,lasso,4234.983621,6123.353679,0.738853,4181.194837,5796.284903,0.783593,0c124c7b6d174953a688117b25d574a5
0,linear_regression,4234.983570,6123.353823,0.738853,4181.194474,5796.284659,0.783593,12f943bc4a4748d1b9604c2c4cc53d44
5,svr,8053.130598,12462.430820,-0.079355,8274.757093,12726.160647,-0.043198,acbdb658d2f440eeafea558a597b44fd


## 13) Mostrar el mejor + guardar una copia en models/

In [45]:
best

{'model': Pipeline(steps=[('preprocess',
                  ColumnTransformer(transformers=[('num', StandardScaler(),
                                                   ['age', 'bmi', 'children']),
                                                  ('cat',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['sex', 'smoker',
                                                    'region'])])),
                 ('model', GradientBoostingRegressor(random_state=42))]),
 'name': 'gbr',
 'cv_rmse': np.float64(4646.314894752352),
 'run_id': 'f3b90d18373445109f15eaee542dd11a'}

In [46]:
import joblib

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

best_path = os.path.join(MODELS_DIR, "best_model.joblib")
joblib.dump(best["model"], best_path)

best_path

'd:\\Maestria DS\\Ciclo III\\C03_MLOps\\Project_example\\mlops-final-project-Jonathan_Sanchez\\models\\best_model.joblib'

## 14) Registrar “best_model” en MLflow con una run final

In [47]:
with mlflow.start_run(run_name="best_model_selection") as run:
    mlflow.log_param("best_model_name", best["name"])
    mlflow.log_param("best_run_id", best["run_id"])
    mlflow.log_metric("best_cv_rmse", float(best["cv_rmse"]))

    mlflow.log_artifact(best_path, artifact_path="exported_model")